## Ximea vs Prime95B — Raw Analysis

Detects and fits 100 nm bead data acquired on two cameras for comparison:

- **Ximea** — Bayer/CFA sensor. Fit with the standard colour pipeline
  (`SR_Functions.fit_QD_data`), giving per-spot `A_B`/`A_G`/`A_R` colour
  fractions from the real Bayer-filtered pixels.
- **Prime95B** — plain mono sensor, no colour filter array. Fit with a single
  intensity channel (`FittingStrategy.NOCOLOUR`) — there is currently no
  high-level wrapper for this in `SR_Functions.py` (its orchestration methods
  — `fit_QD_data`, `fit_SM_data`, `_process_roi`, `_filter_fit_results` — are
  all hard-coded to Bayer masks and `A_B`/`A_G`/`A_R` columns), so this
  notebook builds the mono detect → crop → fit loop directly from the
  lower-level, colour-agnostic building blocks (`SpotDetection_Functions`,
  `Image_Analysis_Functions.fit_puncta_parallel_method`).

**Note:** the `.h5` files already sitting next to the Prime95B `.ome.tif`
files have the *same* `A_B`/`A_G`/`A_R`-column schema as the Ximea results —
i.e. they look like they were produced by forcing the mono data through the
colour pipeline, not a real non-CFA fit. This notebook writes fresh
`*_nocolour_fit.h5` files alongside them rather than overwriting/trusting the
existing ones.

For each camera: load calibration → preview spot detection on one FOV to tune
`pfa`/`sigma`/`fraction_true` → run the full fit across all FOVs.


In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(name)s — %(levelname)s — %(message)s')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import types
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import sys
sys.path.append('../..')

from src import IOFunctions, HelperFunctions, SpotDetectionFunctions, ImageAnalysisFunctions, SR_Functions, sCMOSFunctions
from src import PlottingBase
from src.ImageAnalysisFunctions import FittingStrategy

IO      = IOFunctions.IO_Functions()
HELPER  = HelperFunctions.Helper_Functions()
sCMOS   = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

smoothing_function = types.SimpleNamespace(
    args={'sigma': 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg='image',
)


In [ ]:
# ── Acquisition parameters ─────────────────────────────────────────────────────
DATA_ROOT = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads')
XIMEA_FOLDER    = DATA_ROOT / '100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Ximea'
PRIME95B_FOLDER = DATA_ROOT / '100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Prime95B'

CAL_DIR_XIMEA    = '../../Camera_Calibrations/Ximea_Camera/'
CAL_DIR_PRIME95B = '../../Camera_Calibrations/Prime95B_Camera/'

FIGURE_DIR = Path('./figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NA = 1.49  # same objective/optical train for both camera ports

# Ximea: confirmed elsewhere in the codebase (Camera_Calibrations/, src/CameraDefaults.py).
PIXEL_SIZE_XIMEA_NM = 69.0
# Prime95B: MicroManager metadata has PixelSizeUm=0.0 (never calibrated in MM), and the
# 11 um physical sensor pitch isn't usable without the port magnification — confirmed value.
PIXEL_SIZE_PRIME95B_NM = 110.0

# Representative detection wavelength (only affects the spot-detection matched-filter PSF
# size, not the fit itself). Samples were excited at 488/561/638 nm (see power.txt); this is
# a broad middle-of-range value — adjust per channel if detection needs tuning per laser line.
PEAK_WAVELENGTH_UM = 0.550

PFA      = 1e-4   # spot-detection false-positive rate for the full fits
ROI_size = 16

print(f'Ximea folder    : {XIMEA_FOLDER}')
print(f'Prime95B folder : {PRIME95B_FOLDER}')


In [ ]:
# ── Discover FOVs ───────────────────────────────────────────────────────────────
# Each position is a single self-contained .ome.tif (no multi-file splitting here,
# unlike some other datasets in this repo) with a companion _metadata.txt and,
# in most cases, a pre-existing .h5 (see note above re: Prime95B's being colour-schema).

ximea_files    = sorted(XIMEA_FOLDER.glob('*_MMStack_Pos-*.ome.tif'))
prime95b_files = sorted(PRIME95B_FOLDER.glob('*_MMStack_Pos-*.ome.tif'))

print(f'[Ximea]    {len(ximea_files)} FOVs found')
print(f'[Prime95B] {len(prime95b_files)} FOVs found')


## Ximea (Bayer/CFA) — colour analysis

In [ ]:
# ── Camera calibration ─────────────────────────────────────────────────────────
gain_x      = IO.read_tiff(os.path.join(CAL_DIR_XIMEA, 'gain.tif'))
offset_x    = IO.read_tiff(os.path.join(CAL_DIR_XIMEA, 'offset.tif'))
variance_x  = IO.read_tiff(os.path.join(CAL_DIR_XIMEA, 'variance.tif'))
readnoise_x = IO.read_tiff(os.path.join(CAL_DIR_XIMEA, 'readnoise.tif'))
rqe_x       = IO.read_tiff(os.path.join(CAL_DIR_XIMEA, 'rqe.tif'))

print(f'[Ximea] calibration maps loaded — chip shape: {gain_x.shape}')


In [ ]:
SupRes_X = SR_Functions.SuperRes_Functions(camera='ximea')
print(f'[Ximea] pixel_size={SupRes_X.pixel_size*1000:.1f} nm  mosaic_unit=\n{SupRes_X.mosaic_unit}')


In [ ]:
# ── Spot detection preview — tune parameters before running the full fit ───────
# Detection is performed on the sum of all frames in one example FOV (same
# convention as fit_QD_data), with the demosaiced/variance-aware combined image
# overlaid with detected spot positions. Adjust PFA_PREVIEW/SIGMA_PREVIEW/
# FRACTION_TRUE_PREVIEW and re-run until satisfied, then carry PFA into the
# full-fit cell below.

PFA_PREVIEW           = PFA
SIGMA_PREVIEW         = 3      # Gaussian detection sigma (pixels)
FRACTION_TRUE_PREVIEW = 0.5    # prior on fraction of pixels containing true spots

example_file_x = ximea_files[0]
print(f'[Ximea] Loading {example_file_x.name} ...')
full_x = IO.read_tiff(str(example_file_x), dtype='float32')   # (n_frames, H, W)
n_frames_x = full_x.shape[0]

off_sum_x = offset_x * n_frames_x
var_sum_x = variance_x * n_frames_x
summed_x  = full_x.sum(axis=0)

img_detect_x = SupRes_X._demosaic_image(
    summed_x, use_variance_aware=True,
    gain_map=gain_x, offset_map=off_sum_x, variance=var_sum_x,
)

detected_x = SupRes_X.spot_detection.detect_puncta_in_image(
    img_detect_x,
    pfa=PFA_PREVIEW, variance=var_sum_x, wavelength=PEAK_WAVELENGTH_UM,
    pixel_size=SupRes_X.pixel_size, NA=NA, sigma=SIGMA_PREVIEW,
    fraction_true=FRACTION_TRUE_PREVIEW,
)
print(f'[Ximea] {len(detected_x)} spots detected in {example_file_x.name}')

fig, ax = plt.subplots(figsize=(7, 5.5))
vmin, vmax = np.percentile(img_detect_x, [0.1, 99.9])
ax.imshow(img_detect_x, cmap='gray', vmin=vmin, vmax=vmax, origin='upper', interpolation='nearest')
if len(detected_x):
    ax.scatter(detected_x[:, 1], detected_x[:, 0], s=14, marker='+', color='lime', linewidths=0.8, alpha=0.9)
ax.set_title(f'Ximea — {example_file_x.name}\n{len(detected_x)} spots  (pfa={PFA_PREVIEW:.0e}, σ={SIGMA_PREVIEW})', fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ximea_detection_preview.svg', dpi=300, format='svg')
plt.show()

del full_x
gc.collect()


In [ ]:
# ── Full fit — Ximea ─────────────────────────────────────────────────────────
# fit_QD_data sums all frames per FOV for detection, then fits every individual
# frame at each detected location, saving one <FOV>.h5 per position alongside
# the source .ome.tif. It does not itself skip already-fitted FOVs, so guard
# on existing .h5 count the same way as elsewhere in this repo.

xh5_count = sum(1 for f in os.listdir(XIMEA_FOLDER) if f.endswith('.h5'))
if xh5_count >= len(ximea_files):
    print(f'[Ximea] All {len(ximea_files)} FOV(s) already have .h5 results, skipping fit')
    print('  NOTE: delete the .h5 files and re-run if PFA/ROI_size above changed')
else:
    print(f'[Ximea] Fitting {len(ximea_files)} FOV(s) — this will take a while...')
    SupRes_X.fit_QD_data(
        image_folder=str(XIMEA_FOLDER),
        smoothing_function=smoothing_function,
        gain_map=gain_x,
        offset_map=offset_x,
        rqe=rqe_x,
        read_noise=readnoise_x,
        variance=variance_x,
        n_frames_sum=600,
        pfa=PFA,
        ROI_size=ROI_size,
        peak_wavelength=PEAK_WAVELENGTH_UM,
        NA=NA,
        image_type='.ome.tif',
    )
    print('[Ximea] Done.')


In [ ]:
# ── Quick sanity check ──────────────────────────────────────────────────────────
ximea_h5 = sorted(XIMEA_FOLDER.glob('*.h5'))
ximea_dfs = [pd.read_hdf(str(p), key='data') for p in ximea_h5]
ximea_all = pd.concat(ximea_dfs, ignore_index=True)
print(f'[Ximea] {len(ximea_h5)} FOVs, {len(ximea_all):,} total localisations')
print(ximea_all[['xc', 'yc', 's_x', 's_y', 'A_B', 'A_G', 'A_R', 'photons']].describe())

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.hist(ximea_all['photons'], bins=100)
ax.set_xlabel('Photons')
ax.set_ylabel('Count')
ax.set_title('Ximea — photon distribution')
plt.tight_layout()
plt.show()


## Prime95B (mono, no CFA) — standard analysis

In [ ]:
# ── Camera calibration ─────────────────────────────────────────────────────────
# No rqe map for Prime95B (there's no colour filter array to correct pixel-to-pixel
# spectral efficiency for), so rqe is left as the scalar default (1.0) everywhere below.
gain_p      = IO.read_tiff(os.path.join(CAL_DIR_PRIME95B, 'gain.tif'))
offset_p    = IO.read_tiff(os.path.join(CAL_DIR_PRIME95B, 'offset.tif'))
variance_p  = IO.read_tiff(os.path.join(CAL_DIR_PRIME95B, 'variance.tif'))
readnoise_p = IO.read_tiff(os.path.join(CAL_DIR_PRIME95B, 'readnoise.tif'))

print(f'[Prime95B] calibration maps loaded — chip shape: {gain_p.shape}')


In [ ]:
# ── Spot detection object (mono) ────────────────────────────────────────────────
# SpotDetection_Functions.__init__ validates its camera= argument against
# src.CameraDefaults.CAMERAS, which only registers 'ximea'/'zwo' — there is no
# Prime95B entry. detect_puncta_in_image/detect_puncta_in_stack_parallel are
# themselves camera-agnostic (they just take pixel_size as a plain float), so
# 'ximea' is passed here purely to satisfy that constructor-time validation;
# pixel_size is always overridden explicitly below with the real Prime95B value.
SD_mono = SpotDetectionFunctions.SpotDetection_Functions(camera='ximea', pixel_size=PIXEL_SIZE_PRIME95B_NM / 1000.0)
IAF_mono = ImageAnalysisFunctions.Image_Analysis_Functions()

print(f'[Prime95B] detection pixel_size={SD_mono.pixel_size*1000:.1f} nm')


In [ ]:
# ── Mono (no-CFA) ROI extraction helper ─────────────────────────────────────────
# SuperRes_Functions._process_roi / _process_detected_puncta_batch (src/SR_Functions.py)
# unconditionally index into a 3-channel Bayer mask array, and
# SuperRes_Functions._postprocess_fit_results -> _filter_fit_results hard-codes
# A_B/A_G/A_R columns when filtering — both are Bayer/colour-specific and cannot be
# reused for a mono camera with no colour filter array. This is the mono equivalent:
# crop -> convert to photoelectrons -> smooth -> weight, with no colour mask step,
# matching FittingStrategy.NOCOLOUR's actual (xc, yc, s_x, s_y, bg, A, chi_sqr, frame)
# output columns (verified directly against Image_Analysis_Functions.fit_puncta_method).

NOCOLOUR_FIT_PARAMS  = ['xc', 'yc', 's_x', 's_y', 'bg', 'A', 'chi_sqr', 'frame']
NOCOLOUR_FIT_ERRORS  = ['xc_err', 'yc_err', 's_x_err', 's_y_err', 'bg_err', 'A_err']
NOCOLOUR_ALL_COLUMNS = NOCOLOUR_FIT_PARAMS + NOCOLOUR_FIT_ERRORS


def extract_mono_roi(raw_data, xcentre, ycentre, frame, width, height, ROI_size,
                      smoothing_function, read_noise, gain_map=1.0, offset_map=0.0,
                      rqe=1.0, frame_offset=0, is_multi_frame=False):
    """Crop, photoelectron-convert, smooth and weight one ROI (no colour masks)."""
    bounds = HELPER.calculate_roi_bounds(xcentre, ycentre, ROI_size, width, height)
    if bounds is None:
        return None
    xmin, xmax, ymin, ymax = bounds

    raw_roi = (raw_data[frame, ymin:ymax, xmin:xmax] if is_multi_frame
               else raw_data[ymin:ymax, xmin:xmax])
    if raw_roi.shape[0] != raw_roi.shape[1]:
        return None

    gain_roi   = gain_map[ymin:ymax, xmin:xmax]   if not isinstance(gain_map, (int, float))   else gain_map
    offset_roi = offset_map[ymin:ymax, xmin:xmax] if not isinstance(offset_map, (int, float)) else offset_map
    rqe_roi    = rqe[ymin:ymax, xmin:xmax]        if not isinstance(rqe, (int, float))        else rqe

    photoelectron_roi = IO.convert_to_photoelectrons(raw_roi, gain_map=gain_roi, offset_map=offset_roi, rqe=rqe_roi)

    read_noise_roi = (read_noise[ymin:ymax, xmin:xmax] if not isinstance(read_noise, (int, float))
                       else read_noise)
    smoothed_roi = IO.apply_smoothing(photoelectron_roi, smoothing_function, dtype='float32')
    weights_roi  = IO.generate_weights(smoothed_roi, read_noise=read_noise_roi, dtype='float32')

    return photoelectron_roi, smoothed_roi, weights_roi, (xmin, ymin), frame + frame_offset


def extract_mono_roi_batch(raw_data, detected_puncta, width, height, ROI_size,
                            smoothing_function, read_noise, gain_map=1.0, offset_map=0.0,
                            rqe=1.0, frame_offset=0, is_multi_frame=False):
    """Batch version of extract_mono_roi — mirrors SuperRes_Functions._process_detected_puncta_batch."""
    puncta_tofit, smoothed_tofit, weights_tofit, coords_tofit, planes = [], [], [], [], []
    for i in range(len(detected_puncta)):
        ycentre, xcentre = detected_puncta[i, 0], detected_puncta[i, 1]
        frame = int(detected_puncta[i, 2]) if is_multi_frame else 0
        result = extract_mono_roi(
            raw_data, xcentre, ycentre, frame, width, height, ROI_size,
            smoothing_function, read_noise, gain_map=gain_map, offset_map=offset_map,
            rqe=rqe, frame_offset=frame_offset, is_multi_frame=is_multi_frame,
        )
        if result is None:
            continue
        photoelectron_roi, smoothed_roi, weights_roi, coords, plane = result
        puncta_tofit.append(photoelectron_roi)
        smoothed_tofit.append(smoothed_roi)
        weights_tofit.append(weights_roi)
        coords_tofit.append(coords)
        planes.append(plane)
    return puncta_tofit, smoothed_tofit, weights_tofit, coords_tofit, planes


def filter_nocolour_fits(df, width, height):
    """Mono equivalent of SuperRes_Functions._filter_fit_results (no colour columns)."""
    mask = (
        df.notna().all(axis=1)
        & (df['xc'] > 0) & (df['xc'] < width)
        & (df['yc'] > 0) & (df['yc'] < height)
        & (df['s_x'] > 0) & (df['s_x'] < 3)
        & (df['s_y'] > 0) & (df['s_y'] < 3)
        & (df['A'] > 0)
        & (df['bg'] > 0)
    )
    return df[mask].reset_index(drop=True)


print('Mono ROI-extraction + filtering helpers defined.')


In [ ]:
# ── Spot detection preview — tune parameters before running the full fit ───────
# Same convention as the Ximea preview above: detect on the sum of all frames in
# one example FOV, no demosaicing needed (single intensity channel).

PFA_PREVIEW_P           = PFA
SIGMA_PREVIEW_P         = 3
FRACTION_TRUE_PREVIEW_P = 0.5

example_file_p = prime95b_files[0]
print(f'[Prime95B] Loading {example_file_p.name} ...')
full_p = IO.read_tiff(str(example_file_p), dtype='float32')
n_frames_p = full_p.shape[0]
height_p, width_p = full_p.shape[1], full_p.shape[2]

off_sum_p = offset_p * n_frames_p
var_sum_p = variance_p * n_frames_p
summed_p  = full_p.sum(axis=0)

img_detect_p = IO.convert_to_photoelectrons(summed_p, gain_map=gain_p, offset_map=off_sum_p, rqe=1.0)

detected_p = SD_mono.detect_puncta_in_image(
    img_detect_p,
    pfa=PFA_PREVIEW_P, variance=var_sum_p, wavelength=PEAK_WAVELENGTH_UM,
    pixel_size=SD_mono.pixel_size, NA=NA, sigma=SIGMA_PREVIEW_P,
    fraction_true=FRACTION_TRUE_PREVIEW_P,
)
print(f'[Prime95B] {len(detected_p)} spots detected in {example_file_p.name}')

fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(img_detect_p, [0.1, 99.9])
ax.imshow(img_detect_p, cmap='gray', vmin=vmin, vmax=vmax, origin='upper', interpolation='nearest')
if len(detected_p):
    ax.scatter(detected_p[:, 1], detected_p[:, 0], s=14, marker='+', color='lime', linewidths=0.8, alpha=0.9)
ax.set_title(f'Prime95B — {example_file_p.name}\n{len(detected_p)} spots  (pfa={PFA_PREVIEW_P:.0e}, σ={SIGMA_PREVIEW_P})', fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'prime95b_detection_preview.svg', dpi=300, format='svg')
plt.show()

del full_p
gc.collect()


In [ ]:
# ── Full fit — Prime95B ──────────────────────────────────────────────────────
# For each FOV: detect on the sum of all frames (as above), then fit every
# individual frame at each detected location with FittingStrategy.NOCOLOUR.
# Saved as '<FOV>_nocolour_fit.h5' — deliberately NOT the same filename as the
# pre-existing '<FOV>.h5' files, since those already use the colour-pipeline
# schema (see the note in the title cell) and skipping-if-that-exists would
# wrongly treat them as valid prior NOCOLOUR results.

for i_fov, fov_path in enumerate(prime95b_files):
    out_path = fov_path.with_name(fov_path.stem.replace('.ome', '') + '_nocolour_fit.h5')
    if out_path.exists():
        print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  (skipped, already fitted)')
        continue

    raw = IO.read_tiff(str(fov_path), dtype='float32')   # (n_frames, H, W)
    n_frames = raw.shape[0]
    height, width = raw.shape[1], raw.shape[2]

    off_sum = offset_p * n_frames
    var_sum = variance_p * n_frames
    summed  = raw.sum(axis=0)
    img_detect = IO.convert_to_photoelectrons(summed, gain_map=gain_p, offset_map=off_sum, rqe=1.0)

    detected = SD_mono.detect_puncta_in_image(
        img_detect, pfa=PFA, variance=var_sum, wavelength=PEAK_WAVELENGTH_UM,
        pixel_size=SD_mono.pixel_size, NA=NA, sigma=3, fraction_true=0.2,
    )

    if len(detected) == 0:
        print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  0 spots detected, skipping fit')
        del raw
        gc.collect()
        continue

    # detected_puncta stores [row, col] (2 columns, single-frame detection) — add a
    # constant frame column of 0 so extract_mono_roi_batch's is_multi_frame indexing
    # (which expects a 3rd [row, col, frame] column) still applies per real frame below.
    puncta, smoothed, weights, coords, planes = [], [], [], [], []
    for frame_idx in range(n_frames):
        detected_this_frame = np.column_stack([detected, np.full(len(detected), frame_idx)])
        p, s, w, c, pl = extract_mono_roi_batch(
            raw, detected_this_frame, width, height, ROI_size,
            smoothing_function, readnoise_p, gain_map=gain_p, offset_map=offset_p,
            rqe=1.0, is_multi_frame=True,
        )
        puncta.extend(p); smoothed.extend(s); weights.extend(w); coords.extend(c); planes.extend(pl)

    fit_arr, err_arr = IAF_mono.fit_puncta_parallel_method(
        puncta, smoothed, weights, coords, planes, FittingStrategy.NOCOLOUR,
    )
    df = pd.DataFrame(np.hstack([fit_arr, err_arr]), columns=NOCOLOUR_ALL_COLUMNS)
    df = df.sort_values('frame').reset_index(drop=True)
    df = filter_nocolour_fits(df, width, height)
    # Parity with IOFunctions._add_photon_columns' colour convention
    # (photons = A_B+A_G+A_R, background_photons = bg_B+bg_G+bg_R) — NOCOLOUR
    # only has one channel, so these are just the raw A/bg values renamed.
    df['photons'] = df['A']
    df['background_photons'] = df['bg']

    # write_h5_database already skips _add_photon_columns since 'photons' is
    # present, but still gives us format='table' (matching the existing files'
    # schema) plus NaN-row dropping and frame dtype handling for free.
    IO.write_h5_database(df, str(out_path))

    print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  '
          f'{len(detected)} spots detected, {len(df):,} localisations kept')

    del raw, puncta, smoothed, weights
    gc.collect()

print('[Prime95B] Done.')


In [ ]:
# ── Quick sanity check ──────────────────────────────────────────────────────────
prime95b_h5 = sorted(PRIME95B_FOLDER.glob('*_nocolour_fit.h5'))
prime95b_dfs = [pd.read_hdf(str(p), key='data') for p in prime95b_h5]
prime95b_all = pd.concat(prime95b_dfs, ignore_index=True)
print(f'[Prime95B] {len(prime95b_h5)} FOVs, {len(prime95b_all):,} total localisations')
print(prime95b_all[['xc', 'yc', 's_x', 's_y', 'bg', 'A', 'photons', 'background_photons']].describe())

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.hist(prime95b_all['photons'], bins=100)
ax.set_xlabel('Photons')
ax.set_ylabel('Count')
ax.set_title('Prime95B — photon distribution')
plt.tight_layout()
plt.show()
